In [1]:
import torch as torch
import torch.nn as nn

/Users/andy/Desktop/dev/gpt-from-scratch/venv/lib/python3.14/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


Single head attention block experimentation

## Single-head self-attention

**Goal:** Let each token look at all previous tokens and compute a weighted average of their values based on relevance. One head learns one attention pattern.

**Steps:**
1. In `__init__`: create Wq, Wk, Wv as `nn.Parameter(torch.randn(d_model, d_k))`. Store `self.d_k`.
2. In `forward`: compute Q = X @ Wq, K = X @ Wk, V = X @ Wv
3. Compute scores = QK^T / sqrt(d_k)
4. Create causal mask (lower triangular matrix of ones)
5. Set upper triangle of scores to -inf using mask
6. Apply softmax to get attention weights (each row sums to 1)
7. Output = attention_weights @ V
8. Return output. Shape: (seq_len, d_k)

In [2]:
d_k = 8

corpus = torch.randn(4,8)

In [3]:
corpus.shape

torch.Size([4, 8])

In [4]:
m = corpus.shape[0]
n = corpus.shape[1]
d_model = corpus.shape[1]

Wq = torch.randn(n, d_k)
Wv = torch.randn(n, d_k)
Wk = torch.randn(n, d_k)
    
Q = torch.matmul(corpus, Wq)
V = torch.matmul(corpus, Wv)
K = torch.matmul(corpus, Wk)

attention = torch.matmul(Q, K.T)/(d_k**0.5)

In [5]:
attention

tensor([[  1.8902,  -4.7783,   0.9405,  -4.0483],
        [ 12.7255,  13.0450,   6.5025,   6.5513],
        [ 17.7274,  -1.3198, -25.5170,   7.0653],
        [  1.6746, -11.9502, -15.0616,  -0.6141]])

In [6]:
mask = torch.tril(torch.ones(attention.shape))
mask

tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])

In [7]:
attention.masked_fill(mask == 0, float('-inf'))

tensor([[  1.8902,     -inf,     -inf,     -inf],
        [ 12.7255,  13.0450,     -inf,     -inf],
        [ 17.7274,  -1.3198, -25.5170,     -inf],
        [  1.6746, -11.9502, -15.0616,  -0.6141]])

In [8]:
def self_attention(X, d_k):
    m = X.shape[0]
    n = X.shape[1]

    print("X is", m, "by", n)

    Wq = torch.randn(n, d_k)
    Wv = torch.randn(n, d_k)
    Wk = torch.randn(n, d_k)
    
    Q = torch.matmul(X, Wq)
    V = torch.matmul(X, Wv)
    K = torch.matmul(X, Wk)

    scores = torch.matmul(Q, K.T)/(d_k**0.5)

    mask = torch.tril(torch.ones(scores.shape))

    attention_weights = torch.softmax(scores.masked_fill(mask == 0, float('-inf')), dim=-1)

    attention = torch.matmul(attention_weights, V)

    return attention

In [9]:
self_attention(corpus, 8)

X is 4 by 8


tensor([[ 5.9864, -5.0567, -1.9286,  1.2263,  4.8234,  2.6262, -3.9819, -0.2559],
        [11.6683,  2.1945, -3.4800,  0.5224,  6.7991,  0.5335,  0.1543,  0.5963],
        [11.6659,  2.1914, -3.4793,  0.5227,  6.7983,  0.5344,  0.1526,  0.5959],
        [11.6364,  2.2270, -3.4552,  0.5314,  6.7829,  0.5761,  0.1564,  0.5877]])

In [10]:
self_attention(torch.randn(4, 8), 6)

X is 4 by 8


tensor([[ 2.0135,  1.4807,  0.8104, -1.4327,  0.7322, -4.2896],
        [ 2.0133,  1.4806,  0.8104, -1.4325,  0.7321, -4.2889],
        [ 1.3722,  2.0117,  4.7226,  0.8716, -2.8126,  9.1901],
        [ 0.9704,  1.4727,  3.7022,  0.9952, -2.1973,  8.2240]])

# Single head attention block
### wrapping the attention block in nn.module

In [11]:
class SelfAttention(nn.Module):
    def __init__(self, d_model, d_k):
        super().__init__()
        self.Wk = nn.Parameter(torch.randn(d_model, d_k))
        self.Wq = nn.Parameter(torch.randn(d_model, d_k))
        self.Wv = nn.Parameter(torch.randn(d_model, d_k))
        self.d_k = d_k

    def forward(self, X):
        Q = torch.matmul(X, self.Wq)
        V = torch.matmul(X, self.Wv)
        K = torch.matmul(X, self.Wk)

        scores = torch.matmul(Q, K.transpose(-2, -1))/(self.d_k**0.5)
        mask = torch.tril(torch.ones(scores.shape[-2], scores.shape[-1], device=scores.device))
        attention_weights = torch.softmax(scores.masked_fill(mask == 0, float('-inf')), dim=-1)
        attention = torch.matmul(attention_weights, V)

        return attention

Test case (single head)

In [12]:
attn = SelfAttention(8, 6)
output = attn(torch.randn(4, 8))
print(output.shape)  # should be (4, 6)
output

torch.Size([4, 6])


tensor([[-1.6697,  5.8994,  1.2210,  1.9492,  0.9674,  3.9601],
        [ 5.2876,  1.3007, -2.7709,  1.4176,  2.3427, -4.6088],
        [-1.6686,  5.8986,  1.2204,  1.9491,  0.9677,  3.9586],
        [-3.2994, -3.3584,  2.4348, -2.3339, -1.5605, -2.2594]],
       grad_fn=<MmBackward0>)

## Multi-head attention

**Goal:** Run h single-head attentions in parallel, each with d_k = d_model/h, each learning a different attention pattern. Concatenate and project back.

**Steps:**
1. In `__init__`: create a list of h `SelfAttention` instances, each with d_k = d_model/h. Use `nn.ModuleList`.
2. In `__init__`: create Wo as `nn.Parameter(torch.randn(d_model, d_model))`.
3. In `forward`: run each head on X, getting h outputs each of shape (seq_len, d_k).
4. Concatenate along last dimension: `torch.cat(..., dim=-1)` → (seq_len, d_model).
5. Multiply by Wo → (seq_len, d_model).
6. Return result.

In [13]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, d_k):
        super().__init__()
        # first we find calculate h how many blocks are there
        self.h = int(d_model/d_k)
        self.Wo = nn.Parameter(torch.randn(d_model, d_model))
        self.heads = nn.ModuleList([SelfAttention(d_model, d_k) for _ in range(self.h)])

    def forward(self, X):
        attentions = []
        for head in self.heads:
            attentions.append(head.forward(X))
        attentions = torch.cat(attentions, dim=-1)
        return torch.matmul(attentions, self.Wo)

Test Case (Multi-head)

In [14]:
mha = MultiHeadAttention(d_model=8, d_k=4)  # 8/4 = 2 heads
X = torch.randn(4, 8)  # 4 tokens, 8-dim embeddings
output = mha(X)
print(output.shape)  # should be (4, 8)

torch.Size([4, 8])


## Feedforward network

**Goal:** Each token's representation goes through a small 2-layer neural network independently. Expand to a wider dimension (4x), apply nonlinearity, compress back. Attention is "tokens talking to each other." FFN is "each token thinking alone."

**Steps:**
1. In `__init__`: take `d_model` as argument. Set `d_ff = 4 * d_model`.
2. In `__init__`: create W1 (d_model, d_ff) and W2 (d_ff, d_model) as `nn.Parameter`.
3. In `forward`: hidden = X @ W1
4. Apply `torch.relu(hidden)` — kills all negative values, keeps positive ones unchanged.
5. output = hidden @ W2
6. Return output. Shape: (seq_len, d_model) — same as input.

In [15]:
class FeedForward(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_ff = 4 * d_model # ffn's wider dimension
        self.W1 = nn.Parameter(torch.randn(d_model, self.d_ff)) # map to wider dimension
        self.W2 = nn.Parameter(torch.randn(self.d_ff, d_model)) # map back
    
    def forward(self, X):
        hidden  = torch.matmul(X, self.W1)
        hidden = torch.relu(hidden)
        output = torch.matmul(hidden, self.W2)
        return output


test case (FFN)

In [16]:
ffn = FeedForward(d_model=8)
print(ffn(torch.randn(4, 8)).shape)  # should be (4, 8)

torch.Size([4, 8])


## Transformer block

**Goal:** One block = attention + FFN, connected with residual connections and layer normalization. This is the repeating unit — GPT stacks N of these. Residual connections (x + sublayer(x)) let gradients flow without vanishing. Layer norm stabilizes training.

**Steps:**
1. In `__init__`: take `d_model` and `d_k` as arguments.
2. Create a `MultiHeadAttention` instance.
3. Create an `FFN` instance.
4. Create two layer norms: `self.ln1 = nn.LayerNorm(d_model)` and `self.ln2 = nn.LayerNorm(d_model)`.
5. In `forward`:
   - `x = x + self.mha(self.ln1(x))` (layer norm → attention → add residual)
   - `x = x + self.ffn(self.ln2(x))` (layer norm → FFN → add residual)
   - return x

In [17]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model:int, d_k:int):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, d_k)
        self.ffn = FeedForward(d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = x + self.mha(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

test case (transformer block)

In [18]:
block = TransformerBlock(d_model=8, d_k=4)
print(block(torch.randn(4, 8)).shape)  # should be (4, 8)

torch.Size([4, 8])


## Token + Positional Embedding

**Goal:** Convert raw token IDs (integers) into vectors the transformer can process. Add position information so the model knows word order.

**Steps:**
1. In `__init__`: take `vocab_size`, `d_model`, `max_seq_len` as arguments.
2. Create token embedding: `nn.Embedding(vocab_size, d_model)` — lookup table mapping each token ID to a d_model vector.
3. Create positional embedding: `nn.Embedding(max_seq_len, d_model)` — lookup table mapping each position (0, 1, 2, ...) to a d_model vector.
4. In `forward`: take `x` (shape: seq_len) as integer token IDs.
5. Look up token embeddings: `tok = self.tok_emb(x)` → (seq_len, d_model)
6. Create position indices: `pos_indices = torch.arange(len(x))` → [0, 1, 2, 3]
7. Look up positional embeddings: `pos = self.pos_emb(pos_indices)` → (seq_len, d_model)
8. Return `tok + pos` — add them together, not concatenate.

In [19]:
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model:int, max_seq_len:int):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
    
    def forward(self, X:list[int]):
        tok = self.tok_emb(X)
        pos_indices = torch.arange(X.shape[-1], device=X.device)
        pos = self.pos_emb(pos_indices)

        return tok + pos

In [20]:
emb = TokenEmbedding(vocab_size=100, d_model=8, max_seq_len=32)
x = torch.tensor([5, 12, 3, 77])  # 4 token IDs
print(emb(x).shape)  # should be (4, 8)

torch.Size([4, 8])


# Build the full GPT model. 

It takes token IDs in, produces a probability distribution over the vocabulary for the next token out. It needs to combine everything you've built: embedding → stack of N transformer blocks → layer norm → project back to vocabulary size.

In [21]:
class GPT(nn.Module):
    def __init__(self, vocab_size:int, d_model:int, d_k:int, n_layers:int, max_seq_len:int):
        super().__init__()
        self.vocab_size = vocab_size
        self.n_layers = n_layers
        self.d_model = d_model
        self.d_k = d_k
        self.emb = TokenEmbedding(vocab_size=vocab_size, d_model=d_model, max_seq_len=max_seq_len)
        # first we get a stack of n-transformers
        self.transformers = nn.ModuleList([TransformerBlock(self.d_model, self.d_k) for _ in range(self.n_layers)])
        self.ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
    
    def forward(self, X:list[int]):

        # then we first convert indicies into vectors using dict lookup
        vecs = self.emb(X)

        # each transformer block as a layer
        input = vecs
        for block in self.transformers:
            output = block(input)
            input = output

        output = self.ln(output)
        return self.head(output)
        


In [22]:
model = GPT(vocab_size=100, d_model=8, d_k=4, n_layers=2, max_seq_len=32)
x = torch.tensor([5, 12, 3, 77])
logits = model(x)
print(logits.shape)  # should be (4, 100) — for each token position, a score for every word in vocab

torch.Size([4, 100])


## Training loop

**Goal:** Make the model learn to predict the next character in a text. Feed it chunks of text, measure how wrong its predictions are, update weights, repeat.

**What you need to figure out:**
1. Pick a training text and convert characters to integers (your tokenizer).
2. Grab random chunks of length `max_seq_len` from the text. Input = chunk[:-1], target = chunk[1:].
3. Feed input through the model to get logits. Compute cross-entropy loss against the target.
4. Backprop and update weights.
5. Repeat, printing the loss every N steps. Loss should decrease.
6. After training, feed the model a starting character and let it generate text by predicting one token at a time.

**Useful tools:**
- `torch.optim.Adam(model.parameters(), lr=3e-4)`
- `F.cross_entropy(logits, targets)` — expects logits shape (N, vocab_size) and targets shape (N,)
- `loss.backward()` — computes gradients
- `optimizer.step()` — updates weights
- `optimizer.zero_grad()` — clears old gradients before next iteration

In [23]:
from random import randint

corpus = "Andy is fucking brilliant! " * 1000
chars = set(corpus)
char_to_int = {c:i for i, c in enumerate(chars)}
int_to_char = {i:c for i, c in enumerate(chars)}


In [24]:
vocab_size = len(chars)
print(vocab_size)

18


In [25]:
corpus_ints = [char_to_int[char] for char in corpus]

In [26]:
def get_train(corpus:list[int], n_samples:int, sample_len:int):
    seq_len = len(corpus)
    start_idx = [randint(0, seq_len - sample_len - 1) for _ in range(n_samples)]
    x_train = [corpus[idx:idx+sample_len] for idx in start_idx]
    y_train = [corpus[idx+1:idx+sample_len+1] for idx in start_idx]
    return torch.tensor(x_train), torch.tensor(y_train)

x_train, y_train = get_train(corpus_ints, 5000, 10)
x_train.shape, y_train.shape

(torch.Size([5000, 10]), torch.Size([5000, 10]))

In [27]:
x_train.shape, y_train.shape

(torch.Size([5000, 10]), torch.Size([5000, 10]))

In [28]:
from torch.nn import functional as F

def train(x_train, y_train):
    model = GPT(vocab_size=vocab_size, d_model=8, d_k=4, n_layers=2, max_seq_len=32)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

    for x, y in zip(x_train, y_train):
        logits = model(x)
        cross_entropy = F.cross_entropy(logits, y)
        cross_entropy.backward()
        print(cross_entropy.item())
        optimizer.step()
        optimizer.zero_grad()        

# train(x_train, y_train)

now we generate

In [29]:
def generate(start):
    start_int = [char_to_int[ch] for ch in start]
    logits = model(torch.tensor(start_int))
    # logits[-1].shape
    probs = torch.softmax(logits[-1], dim=-1)
    return torch.multinomial(probs, 1)

now to package this into something reusable

In [30]:
class LLM():
    def __init__(self, corpus, batch_size, sample_len, d_model, d_k, n_layers):
        self.sample_len = sample_len
        self.batch_size = batch_size
        self.corpus = corpus
        chars = set(corpus)
        self.char_to_int = {c:i for i, c in enumerate(chars)}
        self.int_to_char = {i:c for i, c in enumerate(chars)}
        self.corpus_ints = [self.char_to_int[char] for char in corpus]

        self.model = GPT(vocab_size=len(chars), d_model=d_model, d_k=d_k, n_layers=n_layers, max_seq_len=sample_len)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=3e-4)

        self.device = torch.device("mps")
        self.model = self.model.to(self.device)

    def train(self, print_loss=False):
        corpus = self.corpus_ints
        seq_len = len(corpus)
        start_idx = [randint(0, seq_len - self.sample_len - 1) for _ in range(self.batch_size)]
        x_train = [corpus[idx:idx+self.sample_len] for idx in start_idx]
        y_train = [corpus[idx+1:idx+self.sample_len+1] for idx in start_idx]
        x_train, y_train = torch.tensor(x_train), torch.tensor(y_train)

        x_train = x_train.to(self.device)
        y_train = y_train.to(self.device)

        # for x, y in zip(x_train, y_train):
        #     logits = self.model(x)
        #     cross_entropy = F.cross_entropy(logits, y)
        #     if print_loss:
        #         print(cross_entropy)
        #     cross_entropy.backward()
        #     self.optimizer.step()
        #     self.optimizer.zero_grad()        

        logits = self.model(x_train) # (10000, 32, vocab_size)
        # cross_entropy = F.cross_entropy(logits, y_train)
        # cross_entropy.backward()
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y_train.view(-1))
        loss.backward()
        self.optimizer.step()
        self.optimizer.zero_grad()
        return loss.item()

    def generate(self, start, num_chars):
        start_int = [self.char_to_int[ch] for ch in start]
        output = start_int
        for i in range(num_chars):
            # logits = self.model(torch.tensor(output[-self.sample_len:]))
            logits = self.model(torch.tensor(output[-self.sample_len:]).to(self.device))
            probs = torch.softmax(logits[-1], dim=-1)
            output.append(int(torch.multinomial(probs, 1)))
        return ''.join([self.int_to_char[ch] for ch in output])
    

In [31]:
andyGPT = LLM("Andy is a fucking genius " * 5000, 10000, 32, 32, 8, 2)
# andyGPT = LLM("ABCD" * 5000, 10000, 32)
andyGPT.train()
andyGPT.generate("Andy", 100)

'Andyaeiyeu  edsnakigegsykgndi suenAddeaufsa duc kigsfuAec di igidddscfuykgdddyacdnggAaecuenegkcfy seiAee'

let's try with a larger and more proper training corpus

In [32]:
import urllib.request

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "shakespeare.txt")

with open("shakespeare.txt", "r") as f:
    corpus = f.read()

print(len(corpus))  # ~1.1 million characters
print(corpus[:200])  # preview

1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you



With your tiny model, bump up to d_model=64, n_layers=4, d_k=16 and train for 20,000+ steps. You should see it go from random characters to generating plausible-looking (if nonsensical) Shakespearean dialogue.

In [33]:
import time
andyGPT_test = LLM(corpus, batch_size=64, sample_len=32, d_model=64, d_k=16, n_layers=4)
start = time.time()
for i in range(10):
    andyGPT_test.train()
print(f"10 iters: {time.time() - start:.2f}s")

10 iters: 0.96s


In [34]:
andyGPT2 = LLM(corpus, batch_size=512, sample_len=32, d_model=64, d_k=16, n_layers=4)
for i in range(1000):
    loss = andyGPT2.train()
    if i % 100 == 0:
        print(f"{i}: loss={loss:.4f}")


0: loss=4.3479
100: loss=3.3798
200: loss=3.3110
300: loss=3.2952
400: loss=3.2770
500: loss=3.2605
600: loss=3.2447
700: loss=3.2152
800: loss=3.1869
900: loss=3.1753


In [39]:
for i in range(10000):
    loss = andyGPT2.train()
    if i % 100 == 0:
        print(f"{i}: loss={loss:.4f}")

0: loss=3.1337
100: loss=3.1077
200: loss=3.0919
300: loss=3.0646
400: loss=3.0381
500: loss=3.0003
600: loss=2.9788
700: loss=2.9793
800: loss=2.9432
900: loss=2.9330
1000: loss=2.9116
1100: loss=2.9008
1200: loss=2.8942
1300: loss=2.8560
1400: loss=2.8544
1500: loss=2.8495
1600: loss=2.8631
1700: loss=2.8247
1800: loss=2.8205
1900: loss=2.8193
2000: loss=2.7861
2100: loss=2.7940
2200: loss=2.7895
2300: loss=2.7842
2400: loss=2.7494
2500: loss=2.7636
2600: loss=2.7223
2700: loss=2.7093
2800: loss=2.7402
2900: loss=2.7338
3000: loss=2.7224
3100: loss=2.7022
3200: loss=2.6949
3300: loss=2.6989
3400: loss=2.6864
3500: loss=2.6682
3600: loss=2.6565
3700: loss=2.6691
3800: loss=2.6578
3900: loss=2.6601
4000: loss=2.6510
4100: loss=2.6363
4200: loss=2.6266
4300: loss=2.6072
4400: loss=2.6256
4500: loss=2.5996
4600: loss=2.6251
4700: loss=2.6152
4800: loss=2.5928
4900: loss=2.5997
5000: loss=2.5601
5100: loss=2.5826
5200: loss=2.5731
5300: loss=2.5764
5400: loss=2.5699
5500: loss=2.5688
5600

In [41]:
andyGPT2.generate("First", 50)

'Firsts, anar a ulis Lo\n\nWher alis sighy be y wit hocfsa'

In [40]:
for g in andyGPT2.optimizer.param_groups:
    g['lr'] = 1e-3

In [42]:
for i in range(10000):
    loss = andyGPT2.train()
    if i % 100 == 0:
        print(f"{i}: loss={loss:.4f}")

0: loss=2.4078
100: loss=2.4103
200: loss=2.3941
300: loss=2.3862
400: loss=2.3833
500: loss=2.3875
600: loss=2.3613
700: loss=2.3620
800: loss=2.3463
900: loss=2.3451
1000: loss=2.3252
1100: loss=2.3064
1200: loss=2.3126
1300: loss=2.3052
1400: loss=2.2945
1500: loss=2.2763
1600: loss=2.2833
1700: loss=2.2688
1800: loss=2.2816
1900: loss=2.2506
2000: loss=2.2589
2100: loss=2.2519
2200: loss=2.2527
2300: loss=2.2400
2400: loss=2.2434
2500: loss=2.2224
2600: loss=2.2118
2700: loss=2.2156
2800: loss=2.2097
2900: loss=2.2100
3000: loss=2.2135
3100: loss=2.2019
3200: loss=2.1770
3300: loss=2.1815
3400: loss=2.1729
3500: loss=2.1717
3600: loss=2.1391
3700: loss=2.1661
3800: loss=2.1547
3900: loss=2.1457
4000: loss=2.1399
4100: loss=2.1329
4200: loss=2.1493
4300: loss=2.1376
4400: loss=2.1207
4500: loss=2.1244
4600: loss=2.1096
4700: loss=2.1236
4800: loss=2.1046
4900: loss=2.0903
5000: loss=2.1192
5100: loss=2.0894
5200: loss=2.1178
5300: loss=2.0901
5400: loss=2.0588
5500: loss=2.1022
5600

In [ ]:
andyGPT2.generate("First", 50)

'Firstagh I cals, and for brosels? the chay, we lon this'

In [44]:
andyGPT2.generate("Speak", 50)

'Speake, the my loobod:\nHit, thee cod rampost god nimbre'

In [45]:
for i in range(10000):
    loss = andyGPT2.train()
    if i % 100 == 0:
        print(f"{i}: loss={loss:.4f}")

0: loss=1.9075
100: loss=1.9240
200: loss=1.9033
300: loss=1.9121
400: loss=1.9119
500: loss=1.9169
600: loss=1.9032
700: loss=1.9113
800: loss=1.9023
900: loss=1.8855
1000: loss=1.8794
1100: loss=1.8912
1200: loss=1.8860
1300: loss=1.8622
1400: loss=1.8661
1500: loss=1.8690
1600: loss=1.8673
1700: loss=1.8684
1800: loss=1.8597
1900: loss=1.8535
2000: loss=1.8804
2100: loss=1.8550
2200: loss=1.8740
2300: loss=1.8466
2400: loss=1.8453
2500: loss=1.8496
2600: loss=1.8412
2700: loss=1.8630
2800: loss=1.8321
2900: loss=1.8233
3000: loss=1.8495
3100: loss=1.8383
3200: loss=1.8156
3300: loss=1.8346
3400: loss=1.8357
3500: loss=1.7927
3600: loss=1.8165
3700: loss=1.7939
3800: loss=1.8117
3900: loss=1.8257
4000: loss=1.8222
4100: loss=1.8144
4200: loss=1.7947
4300: loss=1.7919
4400: loss=1.8025
4500: loss=1.8067
4600: loss=1.7955
4700: loss=1.8008
4800: loss=1.7959
4900: loss=1.7943
5000: loss=1.8123
5100: loss=1.7939
5200: loss=1.8174
5300: loss=1.7867
5400: loss=1.8071
5500: loss=1.7773
5600

In [46]:
andyGPT2.generate("Speak", 50)

'Speak with madens I do flowelo!\n\nDUKE VINCENTIO:\nA soma'

In [47]:
for j in range(3):
    for i in range(10000):
        loss = andyGPT2.train()
        if i % 100 == 0:
            print(f"{i}: loss={loss:.4f}")
    print("\n\nAfter", j, "training loops of 10,000\n\n")
    andyGPT2.generate("Speak", 50)

0: loss=1.7286
100: loss=1.7414
200: loss=1.7226
300: loss=1.7212
400: loss=1.6880
500: loss=1.7186
600: loss=1.7068
700: loss=1.7008
800: loss=1.6885
900: loss=1.7159
1000: loss=1.7196
1100: loss=1.6863
1200: loss=1.6826
1300: loss=1.7118
1400: loss=1.7105
1500: loss=1.7065


KeyboardInterrupt: 

In [48]:
print(andyGPT2.generate("First Citizen:", 200))


First Citizen:
Where's pret'd the one the beecous fly.
This now: tiscius, whoman, this voity again:
The peoply, I am my long:
Away? and thought the knowes, if no precheabling comfuller, whoste to his?
Hor goold you
